# Coastal Dynamics — Tarragona Coast, Spain
## A Cloud-Native Time Series Analysis of Shoreline Change

**Open Science in Practice — FERS School, Bertinoro, April 2026**

---

### Scientific Question

> Has the coastline near Tarragona (NE Spain) changed in extent and morphology between 2017 and 2024,
> and if so, can those changes be linked to sea level anomaly or sea surface temperature?
> Does the choice of water index (NDWI vs MNDWI) affect the detected change signal?

### Study Site

The study area (`bbox [0.55, 40.55, 1.05, 40.90]`) covers the Tarragona coastal zone
in Catalonia, NE Spain — including the Costa Daurada beaches, the Ebro Delta margin,
and nearshore Mediterranean waters. The area is sensitive to storm surges, longshore
sediment transport, and seasonal wave climatology.

### Methods: two water indices compared

| Index | Formula | Bands | Reference |
|---|---|---|---|
| **NDWI** | (Green − NIR) / (Green + NIR) | B03, B08 | McFeeters (1996) |
| **MNDWI** | (Green − SWIR) / (Green + SWIR) | B03, B11 | Xu (2006) |

MNDWI uses SWIR instead of NIR, which improves separation of water from built-up areas,
sand, and turbid coastal water — making it more reliable for Mediterranean shorelines.
Thresholds are computed automatically per year using Otsu's method instead of a fixed value.

### Data sources (all open, cloud-native)
| Variable | Source | Access |
|---|---|---|
| Multispectral imagery | Sentinel-2 L2A | Planetary Computer STAC |
| Sea Level Anomaly (SLA) | CMEMS SEALEVEL_GLO | `copernicusmarine` Python client |
| Significant Wave Height | CMEMS Wave reanalysis | `copernicusmarine` Python client |

### FAIR compliance
- ✅ All data accessed via public APIs — no local files
- ✅ Notebook fully reproducible on JupyterHub
- ✅ Outputs documented with standard metadata
- 🔄 Results to be published via Zenodo + STAC catalog

---
## Section 0 — Environment Setup

In [ ]:
# Core scientific stack
import numpy as np
import xarray as xr
import rioxarray
import pandas as pd
import geopandas as gpd
import warnings

# STAC access
import pystac_client
import planetary_computer

# CMEMS access
import copernicusmarine

# Raster utilities
from rioxarray.merge import merge_arrays
from rasterio.features import shapes
from shapely.geometry import shape
from scipy.ndimage import binary_opening
from skimage.filters import threshold_otsu

# Visualization
import hvplot.xarray
import hvplot.pandas
import holoviews as hv
import panel as pn
import matplotlib.pyplot as plt
import folium

hv.extension('bokeh')
pn.extension()

print('✅ All imports successful')

---
## Section 1 — Study Area Definition

In [ ]:
# Study area — Tarragona coast, NE Spain
BBOX            = [0.55, 40.55, 1.05, 40.90]   # [lon_min, lat_min, lon_max, lat_max]
YEARS           = list(range(2017, 2026))        # 2017–2025
CLOUD_COVER_MAX = 20                             # % — Mediterranean
ANALYSIS_YEARS  = [2017, 2021, 2024]             # years used for index computation

# Seasonal window: Jan–Apr (spring, low wave energy, consistent lighting)
# Applied to all years to avoid seasonal bias between acquisitions
DATE_WINDOWS = [(f'{y}-01-01', f'{y}-04-30') for y in YEARS]

print(f'Study area bbox: {BBOX}')
print(f'Search window:   Jan–Apr each year')
print(f'Analysis years:  {ANALYSIS_YEARS}')

In [ ]:
# Quick map to verify study area
center_lat = (BBOX[1] + BBOX[3]) / 2
center_lon = (BBOX[0] + BBOX[2]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=10)
folium.Rectangle(
    bounds=[[BBOX[1], BBOX[0]], [BBOX[3], BBOX[2]]],
    color='red', fill=True, fill_opacity=0.2
).add_to(m)
m

---
## Section 2 — Sentinel-2 Scene Search

We search for the least-cloudy scene per tile per year within Jan–Apr.
This seasonal restriction ensures all scenes are comparable and avoids the
November 2017 outlier that caused a systematic NDWI bias in v1.

In [ ]:
# Connect to Planetary Computer STAC
catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace
)
print('✅ Connected to Planetary Computer STAC catalog')

In [ ]:
# Discover which MGRS tiles cover our bbox
discover = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=BBOX,
    datetime='2023-01-01/2023-12-31',
    query={'eo:cloud_cover': {'lt': CLOUD_COVER_MAX}}
)
TILES = sorted({item.properties['s2:mgrs_tile'] for item in discover.items()})
print(f'Tiles covering the study area: {TILES}')

In [ ]:
# Search best scene per tile per year — Jan–Apr window
best_items = {}   # {year: {tile: item}}

for year, (date_start, date_end) in zip(YEARS, DATE_WINDOWS):
    best_items[year] = {}
    for tile in TILES:
        search = catalog.search(
            collections=['sentinel-2-l2a'],
            bbox=BBOX,
            datetime=f'{date_start}/{date_end}',
            query={
                'eo:cloud_cover': {'lt': CLOUD_COVER_MAX},
                's2:mgrs_tile':   {'eq': tile}
            }
        )
        items = list(search.items())
        if items:
            best = min(items, key=lambda x: x.properties['eo:cloud_cover'])
            best_items[year][tile] = best
            print(f"{year}  tile {tile}: {best.datetime.date()} — cloud {best.properties['eo:cloud_cover']:.4f}%")
        else:
            print(f"{year}  tile {tile}: ⚠️  no scenes found — will skip this tile/year")

In [ ]:
# ── VERIFY 2017: confirm scene is Jan–Apr (not November as in v1) ────────────
print('=== 2017 scene check ===')
for tile, item in best_items[2017].items():
    month = item.datetime.month
    flag = '✅' if month in [1, 2, 3, 4] else '⚠️  SEASONAL MISMATCH'
    print(f"  tile {tile}: {item.datetime.date()}  month={month}  {flag}")

# If 2017 still has no Jan–Apr scene, widen cloud threshold for that year only
for tile in TILES:
    if tile not in best_items[2017] or best_items[2017][tile].datetime.month > 4:
        print(f"\n⚠️  Widening cloud threshold for 2017 tile {tile}...")
        search_wide = catalog.search(
            collections=['sentinel-2-l2a'],
            bbox=BBOX,
            datetime='2017-01-01/2017-04-30',
            query={'eo:cloud_cover': {'lt': 50}, 's2:mgrs_tile': {'eq': tile}}
        )
        items_wide = list(search_wide.items())
        if items_wide:
            best_wide = min(items_wide, key=lambda x: x.properties['eo:cloud_cover'])
            best_items[2017][tile] = best_wide
            print(f"  → Replaced with: {best_wide.datetime.date()}  cloud {best_wide.properties['eo:cloud_cover']:.2f}%")
        else:
            print(f"  → No scenes found even at 50% cloud — tile {tile} will be missing for 2017")

---
## Section 3 — Load Bands, Compute NDWI and MNDWI

Both indices are computed for each analysis year:
- **NDWI** (McFeeters 1996): `(B03 − B08) / (B03 + B08)` — Green, NIR at 10m
- **MNDWI** (Xu 2006): `(B03 − B11) / (B03 + B11)` — Green at 10m, SWIR at 20m → reprojected to 10m

Thresholds are computed automatically using **Otsu's method** on the histogram of each scene,
rather than a fixed value of 0. This accounts for scene-to-scene variation in water turbidity,
sun angle, and atmospheric conditions.

In [ ]:
def load_band_mosaic(year, band, overview_level=2):
    """Load a band from all tiles for a year, merge into mosaic, clip to BBOX.
    Re-signs each item immediately before reading to avoid expired SAS tokens.
    """
    tiles_data = []
    for tile, item in best_items[year].items():
        signed = planetary_computer.sign(item)   # fresh token every call
        da = rioxarray.open_rasterio(
            signed.assets[band].href,
            overview_level=overview_level
        ).squeeze()
        tiles_data.append(da)
    mosaic = merge_arrays(tiles_data) if len(tiles_data) > 1 else tiles_data[0]
    return mosaic.rio.clip_box(*BBOX, crs='EPSG:4326')

In [ ]:
ndwi_annual  = {}
mndwi_annual = {}
rgb_annual   = {}
thresholds   = {}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, year in zip(axes, ANALYSIS_YEARS):

    # ── Load bands ───────────────────────────────────────────────────────────
    red   = load_band_mosaic(year, 'B04', overview_level=2)   # 10m Red
    green = load_band_mosaic(year, 'B03', overview_level=2)   # 10m Green
    blue  = load_band_mosaic(year, 'B02', overview_level=2)   # 10m Blue
    nir   = load_band_mosaic(year, 'B08', overview_level=2)   # 10m NIR
    swir  = load_band_mosaic(year, 'B11', overview_level=1)   # 20m SWIR — overview_level=1 ≈ 20m native

    # Reproject SWIR (20m) to match green (10m) spatial grid
    swir_10m = swir.rio.reproject_match(green)

    # Cast to float
    g = green.astype(float)
    n = nir.astype(float)
    s = swir_10m.astype(float)

    # Get source date from first tile
    source_date = str(next(iter(best_items[year].values())).datetime.date())

    # ── NDWI (McFeeters 1996): (Green − NIR) / (Green + NIR) ────────────────
    ndwi = (g - n) / (g + n)
    ndwi.name = 'NDWI'
    ndwi.attrs.update({'long_name': f'NDWI {year}', 'source_date': source_date})

    # ── MNDWI (Xu 2006): (Green − SWIR) / (Green + SWIR) ───────────────────
    mndwi = (g - s) / (g + s)
    mndwi.name = 'MNDWI'
    mndwi.attrs.update({'long_name': f'MNDWI {year}', 'source_date': source_date})

    # ── Otsu thresholds ──────────────────────────────────────────────────────
    ndwi_vals  = ndwi.values.flatten()
    mndwi_vals = mndwi.values.flatten()

    ndwi_clean  = ndwi_vals[np.isfinite(ndwi_vals)]
    mndwi_clean = mndwi_vals[np.isfinite(mndwi_vals)]

    t_ndwi  = threshold_otsu(ndwi_clean)
    t_mndwi = threshold_otsu(mndwi_clean)

    thresholds[year] = {'ndwi': t_ndwi, 'mndwi': t_mndwi}

    ndwi_annual[year]  = ndwi
    mndwi_annual[year] = mndwi
    rgb_annual[year]   = xr.concat([red, green, nir], dim='band')

    # ── RGB visualization ────────────────────────────────────────────────────
    rgb = np.stack([red.values, green.values, blue.values], axis=-1).astype(float)
    p2, p98 = np.percentile(rgb[rgb > 0], [2, 98])
    rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)

    ax.imshow(rgb)
    ax.set_title(f"{year}  |  {source_date}", fontsize=10, fontweight='bold')
    ax.axis('off')

    print(f'✅ {year} ({source_date})')
    print(f'   NDWI  range [{float(ndwi.min()):.2f}, {float(ndwi.max()):.2f}]  Otsu threshold = {t_ndwi:.3f}')
    print(f'   MNDWI range [{float(mndwi.min()):.2f}, {float(mndwi.max()):.2f}]  Otsu threshold = {t_mndwi:.3f}')

plt.suptitle('Sentinel-2 RGB — Tarragona Coast — 2017 / 2021 / 2024', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side NDWI vs MNDWI comparison for each year
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, year in enumerate(ANALYSIS_YEARS):
    t_ndwi  = thresholds[year]['ndwi']
    t_mndwi = thresholds[year]['mndwi']
    date    = ndwi_annual[year].attrs['source_date']

    im0 = axes[0, col].imshow(ndwi_annual[year].values,  cmap='RdYlBu', vmin=-1, vmax=1)
    axes[0, col].set_title(f'NDWI {year}\n{date}  |  threshold = {t_ndwi:.3f}', fontsize=9)
    axes[0, col].axis('off')
    plt.colorbar(im0, ax=axes[0, col], fraction=0.046, pad=0.04)

    im1 = axes[1, col].imshow(mndwi_annual[year].values, cmap='RdYlBu', vmin=-1, vmax=1)
    axes[1, col].set_title(f'MNDWI {year}\n{date}  |  threshold = {t_mndwi:.3f}', fontsize=9)
    axes[1, col].axis('off')
    plt.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)

axes[0, 0].set_ylabel('NDWI', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('MNDWI', fontsize=11, fontweight='bold')

plt.suptitle('NDWI vs MNDWI — Tarragona Coast (blue=water, red=land)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Section 4 — Land Area Quantification

Land pixels are identified as those where the index value is **below** the Otsu threshold
(index < threshold → land/beach; index ≥ threshold → water).

A morphological opening (2 iterations) removes isolated single pixels that are likely noise
rather than real land features.

In [ ]:
PIXEL_AREA_M2 = 100   # 10m × 10m pixel at overview_level=2 (stored at ~20m but computed at native 10m)

land_area    = {}   # {year: {'ndwi_ha': float, 'mndwi_ha': float}}
land_gdf     = {}   # {year: {'ndwi': GeoDataFrame, 'mndwi': GeoDataFrame}}

for year in ANALYSIS_YEARS:
    land_area[year]  = {}
    land_gdf[year]   = {}

    for method, index_data in [('ndwi', ndwi_annual[year]), ('mndwi', mndwi_annual[year])]:

        threshold = thresholds[year][method]

        # Binary land mask with Otsu threshold
        land_mask = (index_data < threshold).astype('uint8')

        # Morphological opening — removes isolated noisy pixels
        land_clean = binary_opening(land_mask.values, iterations=2).astype('uint8')
        land_mask_clean = xr.DataArray(
            land_clean,
            dims=land_mask.dims,
            coords=land_mask.coords
        )
        land_mask_clean.rio.write_crs(index_data.rio.crs, inplace=True)

        # Vectorise land pixels → polygons
        transform = index_data.rio.transform()
        crs       = index_data.rio.crs
        polys = []
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            for geom, val in shapes(land_clean, transform=transform):
                if val == 1:
                    polys.append(shape(geom))

        gdf = gpd.GeoDataFrame(geometry=polys, crs=crs).dissolve()
        land_gdf[year][method] = gdf

        # Area in hectares
        gdf_m   = gdf.to_crs(gdf.estimate_utm_crs())
        area_ha = gdf_m.area.sum() / 10_000
        land_area[year][f'{method}_ha'] = area_ha

    print(f'{year}: NDWI land = {land_area[year]["ndwi_ha"]:.1f} ha  |  MNDWI land = {land_area[year]["mndwi_ha"]:.1f} ha')

# Build comparison DataFrame
df_area = pd.DataFrame([
    {'year': y, 'ndwi_ha': v['ndwi_ha'], 'mndwi_ha': v['mndwi_ha']}
    for y, v in land_area.items()
])
df_area

---
## Section 5 — CMEMS Oceanographic Variables

In [ ]:
# ── Sea Level Anomaly ────────────────────────────────────────────────────────
ds_sla = copernicusmarine.open_dataset(
    dataset_id='cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D',
    variables=['sla'],
    minimum_longitude=BBOX[0],
    maximum_longitude=BBOX[2],
    minimum_latitude=BBOX[1],
    maximum_latitude=BBOX[3],
    start_datetime='2017-01-01',
    end_datetime='2024-12-31',
)

print(ds_sla)

# Annual mean SLA spatially averaged over the Tarragona bbox
sla_annual_mean = (
    ds_sla['sla']
    .mean(dim=['latitude', 'longitude'])
    .resample(time='YE')
    .mean()
)

df_sla = sla_annual_mean.to_dataframe().reset_index()
df_sla['year'] = df_sla['time'].dt.year
df_sla = df_sla[['year', 'sla']]
df_sla

In [ ]:
# ── Significant Wave Height ───────────────────────────────────────────────────
ds_waves = copernicusmarine.open_dataset(
    dataset_id='cmems_mod_glo_wav_my_0.2deg_PT3H-i',
    variables=['VHM0'],
    minimum_longitude=BBOX[0],
    maximum_longitude=BBOX[2],
    minimum_latitude=BBOX[1],
    maximum_latitude=BBOX[3],
    start_datetime='2017-01-01',
    end_datetime='2024-12-31',
)

print(ds_waves)

# Storm season: Oct–Feb (peak Mediterranean wave energy)
hs = ds_waves['VHM0']
hs_storm = hs.sel(time=hs.time.dt.month.isin([10, 11, 12, 1, 2]))

hs_annual = (
    hs_storm
    .mean(dim=['latitude', 'longitude'])
    .resample(time='YE')
    .mean()
)

df_hs = hs_annual.to_dataframe().reset_index()
df_hs['year'] = df_hs['time'].dt.year
df_hs = df_hs[['year', 'VHM0']].rename(columns={'VHM0': 'hs_m'})
df_hs

In [ ]:
# Merge all variables into one DataFrame (analysis years only)
df = (
    df_area
    .merge(df_sla, on='year')
    .merge(df_hs,  on='year')
)
df['year'] = df['year'].astype(int)

print('Integrated dataset:')
df

---
## Section 6 — Interactive Dashboard

The dashboard has two panels:
- **Left:** Index map (NDWI or MNDWI) for the selected year, with land polygons overlaid
- **Right:** Time series of land area (both indices), sea level anomaly, and wave height

Controls:
- **Year slider** — toggles between 2017, 2021, 2024
- **Index toggle** — switches the left map between NDWI and MNDWI

The Otsu threshold used for each year/method is shown in the map title.

In [ ]:
year_slider   = pn.widgets.DiscreteSlider(
    name='Year', options=ANALYSIS_YEARS, value=ANALYSIS_YEARS[0]
)
method_toggle = pn.widgets.RadioButtonGroup(
    name='Index method', options=['NDWI', 'MNDWI'], value='NDWI', button_type='primary'
)

# ── LEFT PANEL: index map + land polygons ────────────────────────────────────
@pn.depends(year_slider.param.value, method_toggle.param.value)
def index_map(year, method):
    data      = ndwi_annual[year]  if method == 'NDWI' else mndwi_annual[year]
    gdf       = land_gdf[year][method.lower()].to_crs('EPSG:4326')
    threshold = thresholds[year][method.lower()]
    date      = data.attrs['source_date']

    base = data.hvplot.image(
        x='x', y='y',
        cmap='RdYlBu', clim=(-1, 1),
        title=f'{method} — Tarragona {year}  |  {date}  |  Otsu threshold = {threshold:.3f}',
        width=540, height=480,
        colorbar=True,
        clabel=f'{method} (blue=water, red=land)',
        geo=True, tiles='OSM',
    )

    polys = gdf.hvplot.polygons(
        geo=True,
        fill_color=None,
        line_color='black',
        line_width=1.5,
    )

    return base * polys


# ── RIGHT PANEL: time series ─────────────────────────────────────────────────
@pn.depends(year_slider.param.value)
def timeseries_panel(year):

    # ── FIX: melt df so hvplot gets one 'method' column to use as groupby ────
    df_melted = df[['year', 'ndwi_ha', 'mndwi_ha']].melt(
        id_vars='year',
        value_vars=['ndwi_ha', 'mndwi_ha'],
        var_name='method',
        value_name='land_ha'
    )
    # Clean labels for legend
    df_melted['method'] = df_melted['method'].map({
        'ndwi_ha':  'NDWI',
        'mndwi_ha': 'MNDWI'
    })

    area_both = (
        df_melted.hvplot.line(
            x='year', y='land_ha', by='method',
            color=['#F5A623', '#8E44AD'],
            line_width=2.5,
            width=420, height=190,
            ylabel='Land area (ha)',
            title='Land area: NDWI vs MNDWI',
        )
        * df_melted.hvplot.scatter(
            x='year', y='land_ha', by='method',
            color=['#F5A623', '#8E44AD'], size=60,
        )
        * hv.VLine(year).opts(color='gray', line_dash='dashed')
    )

    sla_line = (
        df_sla.hvplot.line(
            x='year', y='sla',
            color='#2E86C1', line_width=2.5,
            label='SLA (m)',
            width=420, height=170,
            ylabel='SLA (m)',
            title='Sea level anomaly',
        )
        * df_sla.hvplot.scatter(x='year', y='sla', color='#2E86C1', size=40)
        * hv.VLine(year).opts(color='gray', line_dash='dashed')
    )

    hs_line = (
        df_hs.hvplot.line(
            x='year', y='hs_m',
            color='#1ABC9C', line_width=2.5,
            label='Hs (m)',
            width=420, height=170,
            ylabel='Hs (m)',
            title='Significant wave height (storm season Oct–Feb)',
        )
        * df_hs.hvplot.scatter(x='year', y='hs_m', color='#1ABC9C', size=40)
        * hv.VLine(year).opts(color='gray', line_dash='dashed')
    )

    return pn.Column(
        pn.pane.Markdown('### Time series'),
        area_both,
        sla_line,
        hs_line,
    )


# ── ASSEMBLE DASHBOARD ───────────────────────────────────────────────────────
header = pn.pane.Markdown(
    '# Tarragona Coast — Shoreline Dynamics 2017 / 2021 / 2024\n'
    '**Data:** Sentinel-2 L2A (Planetary Computer) · CMEMS SLA · CMEMS Wave reanalysis  '
    '| **Indices:** NDWI (McFeeters 1996) vs MNDWI (Xu 2006) | **Threshold:** Otsu (per year)',
    sizing_mode='stretch_width'
)

controls = pn.Row(
    pn.Column('### Year', year_slider),
    pn.Column('### Index method', method_toggle),
)

dashboard = pn.Column(
    header,
    controls,
    pn.Row(
        pn.panel(index_map),
        pn.panel(timeseries_panel),
    ),
)

dashboard.servable()
dashboard

---
## Section 7 — Interpretation

> ⚠️ *Fill this section after running the analysis with real data.*

### Index comparison
```
- NDWI Otsu thresholds ranged from ___ to ___ across years.
- MNDWI Otsu thresholds ranged from ___ to ___.
- MNDWI detected [more / less / similar] land area than NDWI in all years.
- The largest discrepancy between indices was in ___ (year), likely because ___.
```

### Temporal change
```
- Land area changed from ___ ha (2017) to ___ ha (2024) using MNDWI — a ___% change.
- The direction of change is consistent / inconsistent between NDWI and MNDWI.
- Years with higher SLA (___) correspond to [larger / smaller] land area estimates.
- Wave height was highest in ___, which may explain ___.
```

### Limitations
- Even within Jan–Apr, inter-annual variation in tidal state, sun angle, and
  atmospheric conditions can affect index values independently of real shoreline change.
- At overview_level=2 (~20m effective resolution), changes smaller than ~40m are below
  the detection limit.
- Only 3 time points — a longer series would be needed to detect trends statistically.

### FAIR statement
| Principle | Status |
|---|---|
| **F**indable | ✅ All data discoverable via STAC and CMEMS catalog APIs |
| **A**ccessible | ✅ Sentinel-2 via Planetary Computer (free); CMEMS (free account) |
| **I**nteroperable | ✅ CF-compliant coordinates via rioxarray throughout |
| **R**eusable | ✅ Notebook reproducible on any JupyterHub with Pangeo stack |